# 1. Baseline Model: TF-IDF + Logistic Regression for Sarcasm Detection
**CSE 4122 — Natural Language Processing Laboratory**  
*Department of Computer Science and Engineering, Khulna University of Engineering & Technology (KUET)*

---

### Overview
This notebook implements the traditional machine learning baseline for sarcasm detection:
- **Feature Representation**: Term Frequency-Inverse Document Frequency (TF-IDF) with unigram and bigram n-grams.
- **Classifier**: $L_2$-regularized Logistic Regression with balanced class weighting to handle label skew.
- **Explainability**: Feature importance analysis via top positive and negative regression coefficients.


## 1. Setup & Environment
Import essential scientific, text-processing, and evaluation libraries.

In [ ]:
import os
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, f1_score,
    classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve
)

# Set random seeds and styling
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
print("[OK] Libraries successfully loaded.")


## 2. Dataset Loading & Inspection
Load the official training dataset (`train.csv`) and test dataset (`test_1.csv`).

In [ ]:
# Locate dataset paths
train_path = os.path.join("dataset", "train.csv")
test_path = os.path.join("dataset", "test_1.csv")

if not os.path.exists(train_path):
    train_path = "train.csv"
if not os.path.exists(test_path):
    test_path = "test_1.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Normalize column names
text_col_train = "tweet" if "tweet" in train_df.columns else "text"
text_col_test = "tweet" if "tweet" in test_df.columns else "text"

train_df["text_clean"] = train_df[text_col_train].fillna("").astype(str)
test_df["text_clean"] = test_df[text_col_test].fillna("").astype(str)

y_train = train_df["sarcastic"].astype(int).values
y_test = test_df["sarcastic"].astype(int).values

print(f"Train samples: {len(train_df)} | Sarcastic: {y_train.sum()} ({y_train.mean()*100:.1f}%) | Non-Sarcastic: {len(y_train) - y_train.sum()}")
print(f"Test samples:  {len(test_df)}  | Sarcastic: {y_test.sum()} ({y_test.mean()*100:.1f}%)  | Non-Sarcastic: {len(y_test) - y_test.sum()}")


## 3. Sarcasm-Aware Text Preprocessing
Social media sarcasm frequently relies on punctuation (quotes, question marks, exclamation marks), emojis, and capitalization. We apply normalized text cleaning while preserving these informative signals.

In [ ]:
def clean_text_for_tfidf(text: str) -> str:
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)  # remove URLs
    text = re.sub(r'@[A-Za-z0-9_]+', ' ', text)          # remove mentions
    text = text.replace('&amp;', '&').replace('&lt;', '<').replace('&gt;', '>')
    # Preserve key punctuation cues like quotes, exclamation, question marks
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'\"-]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

X_train_raw = [clean_text_for_tfidf(t) for t in train_df["text_clean"]]
X_test_raw = [clean_text_for_tfidf(t) for t in test_df["text_clean"]]

print("Example Raw :", train_df["text_clean"].iloc[0])
print("Example Clean:", X_train_raw[0])

## 4. TF-IDF Feature Extraction
We extract unigram and bigram features with sublinear term-frequency scaling ($1 + \log(TF)$) and document frequency filtering.

In [ ]:
tfidf = TfidfVectorizer(
    max_features=8000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

X_train_tfidf = tfidf.fit_transform(X_train_raw)
X_test_tfidf = tfidf.transform(X_test_raw)

print(f"TF-IDF Matrix Shape (Train): {X_train_tfidf.shape}")
print(f"TF-IDF Matrix Shape (Test):  {X_test_tfidf.shape}")


## 5. Model Training: Logistic Regression
We train Logistic Regression with `class_weight='balanced'` to offset the 3:1 non-sarcastic to sarcastic imbalance in the training data.

In [ ]:
lr_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    class_weight="balanced",
    random_state=SEED
)

lr_model.fit(X_train_tfidf, y_train)
print("[OK] Logistic Regression model successfully trained.")


## 6. Evaluation on Test Set
Evaluate accuracy, precision, recall, F1-score, and macro-F1 on the official test set.

In [ ]:
y_pred = lr_model.predict(X_test_tfidf)
y_prob = lr_model.predict_proba(X_test_tfidf)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)
macro_f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)

print("="*45)
print("  TF-IDF + Logistic Regression Test Metrics")
print("="*45)
print(f"Accuracy:   {acc:.4f}")
print(f"Precision:  {prec:.4f}")
print(f"Recall:     {rec:.4f}")
print(f"F1 Score:   {f1:.4f}")
print(f"Macro-F1:   {macro_f1:.4f}")
print(f"ROC-AUC:    {roc_auc:.4f}")
print("="*45)
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=["Non-Sarcastic", "Sarcastic"]))


## 7. Confusion Matrix & ROC Curve Visualization

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[0],
            xticklabels=["Non-Sarcastic", "Sarcastic"],
            yticklabels=["Non-Sarcastic", "Sarcastic"])
axes[0].set_title("Confusion Matrix — TF-IDF + LR")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
axes[1].plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
axes[1].plot([0, 1], [0, 1], color="navy", lw=1.5, linestyle="--")
axes[1].set_xlabel("False Positive Rate")
axes[1].set_ylabel("True Positive Rate")
axes[1].set_title("Receiver Operating Characteristic (ROC)")
axes[1].legend(loc="lower right")

plt.tight_layout()
plt.show()


## 8. Feature Explainability: Top Sarcastic vs Non-Sarcastic Words
Logistic regression coefficients directly reveal which n-grams push predictions toward sarcasm ($+$ coef) versus literal speech ($-$ coef).

In [ ]:
feature_names = np.array(tfidf.get_feature_names_out())
coefs = lr_model.coef_[0]

top_k = 15
top_sarcastic_idx = np.argsort(coefs)[-top_k:]
top_literal_idx = np.argsort(coefs)[:top_k]

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(feature_names[top_sarcastic_idx], coefs[top_sarcastic_idx], color="coral")
axes[0].set_title("Top 15 Features Indicating SARCASM")
axes[0].set_xlabel("Logistic Regression Weight (+)")

axes[1].barh(feature_names[top_literal_idx], coefs[top_literal_idx], color="skyblue")
axes[1].set_title("Top 15 Features Indicating NON-SARCASM")
axes[1].set_xlabel("Logistic Regression Weight (-)")

plt.tight_layout()
plt.show()


## 9. Interactive Sarcasm Predictor
Test custom social media sentences using the trained TF-IDF + Logistic Regression baseline.

In [ ]:
def predict_sarcasm(text: str):
    cleaned = clean_text_for_tfidf(text)
    vec = tfidf.transform([cleaned])
    prob = lr_model.predict_proba(vec)[0][1]
    verdict = "SARCASTIC" if prob >= 0.5 else "NON-SARCASTIC"
    print(f"Input:   \"{text}\"")
    print(f"Verdict: {verdict} (Confidence: {prob if prob >= 0.5 else 1 - prob:.2%})")
    print("-" * 50)

# Demonstration examples
predict_sarcasm("Oh wonderful, another software update right in the middle of my presentation!")
predict_sarcasm("The library will be open tomorrow from 9am to 5pm.")
predict_sarcasm("I absolutely love standing in line for two hours in the rain.")